# 데이터 EDA 및 모델 학습 — 실습 따라하기`(실습-문제) 1-1` / `(과제-문제) 1-1` 의 전 과정을 **한 번에 훑는 코드 노트북**입니다.빈칸 없이 완성된 코드이므로 셀을 위에서부터 실행하며 결과를 눈으로 확인하세요.| Step | 내용 ||---|---|| 0 | 환경 준비 · 데이터 로딩 || 1 | EDA — 통계량, 상관관계, 분포, scatter/pairplot || 2 | 결측치 · 이상치 (IQR) || 3 | 전처리 — train_test_split, StandardScaler, 데이터 누수 || 4 | 분류 — LogisticRegression, 혼동행렬, classification_report, ROC-AUC || 5 | 교차검증 || 6 | PCA + KMeans || 7 | 회귀 — LinearRegression, RMSE/MAE/R² || 8 | Pipeline으로 누수 원천 차단 |> **참고**: `(과제-문제)`는 Boston 데이터를 `fetch_openml`로 받아오지만 인터넷이 필요합니다.> 이 노트북에서는 **오프라인에서도 항상 동작**하도록 Boston과 동일한 구조의 합성 주택 데이터를 만들어 씁니다.> 실제 Boston을 쓰고 싶으면 Step 7의 주석 처리된 코드를 사용하세요.

## Step 0. 환경 준비 및 데이터 로딩

In [ ]:
# 최초 1회만 실행 (이미 설치되어 있으면 건너뛰어도 됩니다)# !pip install -q "numpy>=2.0.0" "pandas>=2.0.0" "matplotlib>=3.8.0" "seaborn>=0.13.2" "scipy>=1.13.0" "scikit-learn>=1.4.2" 

In [ ]:
import numpy as npimport pandas as pdimport matplotlibimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme()for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:    try:        matplotlib.rc("font", family=f); break    except Exception:        passmatplotlib.rcParams["axes.unicode_minus"] = Falsefrom sklearn.datasets import load_wine# 데이터 불러오기 — 타깃 y를 'quality' 컬럼으로 붙여둔다df, y = load_wine(as_frame=True, return_X_y=True)df["quality"] = yprint("shape:", df.shape)df.head()

---## Step 1. 탐색적 데이터 분석 (EDA)### 1-1. 데이터 파헤치기 — pandas 기본기

In [ ]:
# 1. 총 샘플 수sample_count = df.shape[0]                       # len(df) 도 가능# 2. 컬럼 수 (타깃 포함)feature_count = df.shape[1]# 3. 타깃 클래스 개수class_count = df["quality"].nunique()# 4. 클래스별 샘플 수class_distribution = df["quality"].value_counts()# 5. Alcohol 평균이 가장 높은 클래스 (idxmax는 '값'이 아니라 '인덱스'를 준다!)top_alcohol_class = df.groupby("quality")["alcohol"].mean().idxmax()# 6. malic_acid 평균malic_mean = df["malic_acid"].mean()print("샘플 수      :", sample_count)print("컬럼 수      :", feature_count)print("클래스 개수  :", class_count)print("클래스 분포  :"); print(class_distribution.sort_index())print("alcohol 평균 최고 클래스:", top_alcohol_class)print("malic_acid 평균:", round(malic_mean, 6))

In [ ]:
# 요약 통계 한 번에 (transpose 하면 보기 편하다)df.describe().T[["count", "mean", "std", "min", "50%", "max"]]

### 1-2. 상관관계 히트맵Pearson 상관계수는 **선형** 관계를 -1 ~ 1로 나타냅니다.⚠️ `quality`는 원래 범주형이지만 0/1/2가 연속형처럼 들어가 있어 계산이 됩니다. 해석에 주의하세요.

In [ ]:
corr = df.corr(numeric_only=True)fig, ax = plt.subplots(figsize=(11, 8))n = len(corr)mask = np.triu(np.ones((n, n)))     # 대각선 포함 위쪽 가리기 (자가상관 1 + 대칭 중복 제거)ax.grid(False)                       # 가려진 뒤의 격자 제거ax.set_title("Correlation between features")sns.heatmap(data=corr, annot=True, fmt=".2f", cmap="coolwarm",            mask=mask, annot_kws={"size": 7}, ax=ax)plt.show()print("quality와 상관 |r| 상위 6개")print(corr["quality"].abs().sort_values(ascending=False).head(6))

### 1-3. 분포 시각화 (histplot)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4.6), ncols=3)# 1) 가로 방향 히스토그램 → x 대신 y 를 넘긴다sns.histplot(data=df, y="flavanoids", bins=20, kde=True, ax=ax[0])ax[0].set_title("1) horizontal")# 2) 클래스별 분포 → huesns.histplot(data=df, x="flavanoids", hue="quality", bins=20, kde=True, ax=ax[1])ax[1].set_title("2) hue='quality'")# 3) 두 변수 동시 (2D 히스토그램) → x, y 둘 다 지정sns.histplot(data=df, x="flavanoids", y="total_phenols", bins=20, ax=ax[2])ax[2].set_title("3) bivariate")fig.tight_layout(); plt.show()

### 1-4. scatterplot / pairplot**해석 힌트**: 색(quality)이 구간별로 분리되면 그 두 변수는 클래스 구분에 유용한 신호입니다.

In [ ]:
plt.figure(figsize=(6, 4))sns.scatterplot(data=df, x="flavanoids", y="total_phenols", hue="quality")plt.title("flavanoids vs total_phenols")plt.show()

In [ ]:
# quality와 상관이 가장 높은 5개 특성 (+ quality 자신) → 6개 인덱싱top_features = corr["quality"].abs().sort_values(ascending=False).index[:6]print("선택된 컬럼:", list(top_features))sns.pairplot(data=df[top_features], hue="quality", corner=True, height=1.6)plt.show()

---## Step 2. 결측치와 이상치실무 데이터는 늘 지저분합니다. 실습용으로 결측치/이상치를 **일부러 만들어** 넣고 처리해 봅니다.

In [ ]:
df_missing = df.copy()# 결측치: flavanoids 일부를 NaN 으로np.random.seed(42)missing_idx = np.random.choice(df_missing.index, size=10, replace=False)df_missing.loc[missing_idx, "flavanoids"] = np.nan# 이상치: alcohol 일부를 비정상적으로 크게outlier_idx = np.random.choice(df_missing.index, size=5, replace=False)df_missing.loc[outlier_idx, "alcohol"] = df_missing["alcohol"].mean() * 5print(df_missing.isnull().sum()[lambda s: s > 0])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6), ncols=2)sns.heatmap(df_missing.isnull(), cbar=False, ax=ax[0])ax[0].set_title("Visualize Missing Values")     # 밝은 가로줄 = 결측 위치sns.boxplot(x=df_missing["alcohol"], ax=ax[1])ax[1].set_title("Outliers of Alcohol")plt.tight_layout(); plt.show()

### 2-1. IQR 기반 이상치 탐지$$\mathrm{IQR}=Q_3-Q_1,\qquad \text{정상 범위}=[\,Q_1-1.5\,\mathrm{IQR},\ Q_3+1.5\,\mathrm{IQR}\,]$$

In [ ]:
def detect_outliers_iqr(data, column, k=1.5):    Q1 = data[column].quantile(0.25)    Q3 = data[column].quantile(0.75)    IQR = Q3 - Q1    lower_bound = Q1 - k * IQR    upper_bound = Q3 + k * IQR    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]outliers_alcohol = detect_outliers_iqr(df_missing, "alcohol")print(f"alcohol 이상치 개수: {len(outliers_alcohol)}")print(outliers_alcohol["alcohol"].round(3).tolist())

### 2-2. 결측치 대체 · 이상치 제거- 결측치 처리: 삭제 / 대체(평균·중앙값·최빈값·모델 예측)- 이상치 처리: 삭제 / 변환(log, sqrt) / 경계값 대체- **중요**: 무조건 지우기 전에 *왜 생겼는지* 먼저 확인!

In [ ]:
# 1) 결측치를 평균으로 대체df_filled = df_missing.fillna(df_missing.mean(numeric_only=True))assert df_filled["flavanoids"].isnull().sum() == 0# 2) 이상치 행 제거df_no_outliers = df_filled[~df_filled.index.isin(outliers_alcohol.index)]print(f"처리 전       : {df_missing.shape}")print(f"이상치 제거 후 : {df_no_outliers.shape}")# 평균 대체 vs 중앙값 대체 비교mean_fill = df_missing["flavanoids"].fillna(df_missing["flavanoids"].mean())med_fill  = df_missing["flavanoids"].fillna(df_missing["flavanoids"].median())print(f"\n원본 std       : {df['flavanoids'].std():.4f}")print(f"평균 대체 std   : {mean_fill.std():.4f}   ← 대체하면 분산이 줄어든다")print(f"중앙값 대체 std : {med_fill.std():.4f}")

### 2-3. sklearn 방식 — SimpleImputer`fit`은 train에서만! (Step 3의 데이터 누수 파트에서 다시 설명)

In [ ]:
from sklearn.impute import SimpleImputerimputer = SimpleImputer(strategy="median")sample = df_missing[["flavanoids"]]print("대체 전 결측:", int(sample.isnull().sum().iloc[0]))print("대체 후 결측:", int(pd.DataFrame(imputer.fit_transform(sample)).isnull().sum().iloc[0]))print("사용된 통계값(중앙값):", imputer.statistics_)

---## Step 3. 전처리 — 분할과 표준화### 3-1. train_test_split- `test_size=0.3` : 테스트 30%- `random_state=42` : 결과 고정(재현성)- `stratify=y` : 클래스 비율 유지 → "훈련에 전부 0, 검증에 전부 1" 방지

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler# 이번엔 이진 분류로 단순화: 클래스 0 vs 나머지X = df.drop("quality", axis=1).values.copy()y = df["quality"].values.copy()y = (y != 0).astype(int)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.3, random_state=42, stratify=y)print("train:", X_train.shape, " test:", X_test.shape)print("train 양성 비율:", y_train.mean().round(4))print("test  양성 비율:", y_test.mean().round(4), " ← stratify 덕분에 거의 같다")

### 3-2. StandardScaler — 반드시 train으로만 fit$$z=\frac{x-\mu}{\sigma}\quad(\mu,\sigma\ \text{는 train 통계})$$❌ 전체 데이터로 스케일링 → 분할 &nbsp;&nbsp; ✅ 분할 → train으로 fit → 각각 transform

In [ ]:
scaler = StandardScaler()scaler.fit(X_train)                        # ← train 통계만 학습X_train_norm = scaler.transform(X_train)X_test_norm  = scaler.transform(X_test)    # ← test는 transform만!print("train 평균(≈0):", np.round(X_train_norm.mean(axis=0)[:5], 6))print("train 표준편차(≈1):", np.round(X_train_norm.std(axis=0)[:5], 6))print("test  평균(0이 아님이 정상):", np.round(X_test_norm.mean(axis=0)[:5], 4))

### 3-3. 데이터 누수가 성능을 부풀리는 실험**아무 정보도 없는 무작위 특성**을 잔뜩 만들고,① 전체 데이터로 특성을 고른 뒤 분할 (누수)  vs  ② train으로만 특성을 고름 (정상)을 비교합니다. 기대 정확도는 둘 다 0.5여야 정상입니다.

In [ ]:
from sklearn.neighbors import KNeighborsClassifierdef leakage_demo(n=60, p=300, trials=20, seed=0):    rs = np.random.default_rng(seed)    leak, ok = [], []    for _ in range(trials):        Xr = rs.normal(size=(n, p))          # 라벨과 아무 관계 없는 무작위 특성        yr = np.array([0, 1] * (n // 2))        idx = rs.permutation(n); nte = int(n * 0.3)        te, tr = idx[:nte], idx[nte:]        def top5(rows):            c = np.array([abs(np.corrcoef(Xr[rows, j], yr[rows])[0, 1]) for j in range(p)])            return np.argsort(-c)[:5]        def acc(feats):            m = KNeighborsClassifier(1).fit(Xr[np.ix_(tr, feats)], yr[tr])            return m.score(Xr[np.ix_(te, feats)], yr[te])        leak.append(acc(top5(idx)))          # ❌ 전체로 특성 선택        ok.append(acc(top5(tr)))             # ✅ train으로만    return np.mean(leak), np.mean(ok)l, o = leakage_demo()print(f"❌ 누수 있음 : 평균 정확도 {l:.3f}")print(f"✅ 정상      : 평균 정확도 {o:.3f}")print("→ 정보가 전혀 없는데도 누수가 있으면 성능이 부풀려집니다.")

---## Step 4. 분류 모델 학습 및 평가sklearn 파이프라인: **① 인스턴스 선언 → ② `.fit()` → ③ `.predict()` / `.transform()`**

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import confusion_matrix, classification_reportclf = LogisticRegression(max_iter=1000)     # ConvergenceWarning 나면 max_iter↑ 또는 표준화 확인clf.fit(X_train_norm, y_train)y_pred = clf.predict(X_test_norm)print("Confusion Matrix  [[TN FP],[FN TP]]")print(confusion_matrix(y_test, y_pred))print()print(classification_report(y_test, y_pred, digits=4))

### 4-1. ROC 곡선과 AUC⚠️ `predict_proba` 에 넣는 데이터도 **학습과 동일한 전처리**를 거쳐야 합니다 (`X_test_norm`).

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_scorey_score = clf.predict_proba(X_test_norm)[:, 1]     # 양성(1) 클래스 확률fpr, tpr, thresholds = roc_curve(y_test, y_score)auc = roc_auc_score(y_test, y_score)plt.figure(figsize=(5, 4))plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.3f}")plt.plot([0, 1], [0, 1], linestyle="--", label="Random")plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")plt.title("ROC Curve"); plt.legend(); plt.tight_layout(); plt.show()# 흔한 실수: 전처리 안 된 원본을 넣으면?auc_wrong = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])print(f"올바른 AUC (X_test_norm): {auc:.4f}")print(f"잘못된 AUC (X_test 원본) : {auc_wrong:.4f}   ← 값이 달라진다!")

### 4-2. 임계값(threshold)을 바꾸면 지표가 어떻게 변할까

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_scorerows = []for th in [0.2, 0.35, 0.5, 0.65, 0.8]:    yp = (y_score >= th).astype(int)    rows.append({"threshold": th,                 "accuracy": accuracy_score(y_test, yp),                 "precision": precision_score(y_test, yp, zero_division=0),                 "recall": recall_score(y_test, yp),                 "f1": f1_score(y_test, yp)})print(pd.DataFrame(rows).round(4).to_string(index=False))print("\n임계값↓ → Recall↑ Precision↓ / 임계값↑ → Precision↑ Recall↓")

---## Step 5. 교차검증 (Cross-Validation)한 번의 분할은 운에 좌우됩니다. 여러 번 나눠 평균 내면 일반화 성능을 안정적으로 추정할 수 있습니다.

In [ ]:
from sklearn.model_selection import cross_val_scoref1_scores = cross_val_score(clf, X_train_norm, y_train, cv=5, scoring="f1")print("각 fold F1 :", np.round(f1_scores, 4))print("평균 F1    :", round(f1_scores.mean(), 4))print("표준편차   :", round(f1_scores.std(), 4))for k in [2, 3, 5, 10]:    s = cross_val_score(clf, X_train_norm, y_train, cv=k, scoring="f1")    print(f"k={k:2d}  평균 {s.mean():.4f}  표준편차 {s.std():.4f}")

---## Step 6. 비지도학습 — PCA + KMeans

In [ ]:
from sklearn.decomposition import PCAfrom sklearn.cluster import KMeanspca = PCA(n_components=2)X_pca = pca.fit_transform(X_train_norm)        # 표준화된 데이터에 적용하는 것이 일반적print("explained_variance_ratio_:", np.round(pca.explained_variance_ratio_, 4))print("두 축이 설명하는 분산 합 :", round(pca.explained_variance_ratio_.sum(), 4))kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)y_cluster = kmeans.fit_predict(X_pca)fig, ax = plt.subplots(figsize=(9.5, 3.6), ncols=2)fig.suptitle("KMeans Clustering with PCA")for label in np.unique(y_cluster):    m = y_cluster == label    ax[0].scatter(X_pca[m, 0], X_pca[m, 1], label=label, alpha=.7)ax[0].set_title("Labels inferred by K-Means"); ax[0].legend()for label in np.unique(y_train):    m = y_train == label    ax[1].scatter(X_pca[m, 0], X_pca[m, 1], label=label, alpha=.7)ax[1].set_title("Actual Labels on PCA"); ax[1].legend()fig.tight_layout(); plt.show()print("\n⚠️ K-Means 라벨 번호는 실제 클래스 번호와 대응하지 않습니다.")print("   색이 바뀌어도 '분리 방향'이 같으면 잘 잡은 것입니다.")

---## Step 7. 회귀 — 주택 가격 예측`(과제-문제)`의 Boston Housing 흐름입니다.아래는 **오프라인에서도 돌아가도록** Boston과 같은 구조로 만든 합성 데이터입니다.(실제 Boston을 쓰려면 주석의 `fetch_openml` 코드를 사용하세요.)

In [ ]:
# --- 실제 Boston을 쓰려면 (인터넷 필요) -------------------------------# from sklearn.datasets import fetch_openml# boston = fetch_openml(data_id=531, as_frame=True)# df_h = boston.frame# df_h.columns = [c.upper() for c in df_h.columns]# df_h = df_h.drop("B", axis=1)# ---------------------------------------------------------------------rs = np.random.default_rng(7)n = 506CRIM    = rs.exponential(3.6, n)ZN      = rs.choice([0, 0, 0, 12.5, 20, 35, 80], n).astype(float)INDUS   = rs.uniform(0.5, 27, n)CHAS    = (rs.random(n) < 0.07).astype(int)NOX     = 0.38 + 0.008 * INDUS + rs.normal(0, 0.03, n)RM      = rs.normal(6.28, 0.70, n)AGE     = np.clip(rs.normal(68, 28, n), 2, 100)DIS     = np.clip(rs.lognormal(1.19, 0.42, n), 1, 13)RAD     = rs.choice([1,2,3,4,5,6,7,8,24], n)TAX     = 200 + 12 * RAD + rs.normal(0, 30, n)PTRATIO = np.clip(rs.normal(18.5, 2.2, n), 12, 22)LSTAT   = np.clip(38 - 4.4 * RM + rs.normal(0, 4.5, n), 1.7, 38)MEDV = np.clip(-8 + 8.4*RM - 0.63*LSTAT - 0.18*CRIM - 15*NOX               + 3.1*CHAS - 0.85*PTRATIO + 0.9*DIS + rs.normal(0, 3.0, n), 5, 50)df_h = pd.DataFrame({"CRIM":CRIM,"ZN":ZN,"INDUS":INDUS,"CHAS":CHAS,"NOX":NOX,"RM":RM,                     "AGE":AGE,"DIS":DIS,"RAD":RAD,"TAX":TAX,"PTRATIO":PTRATIO,                     "LSTAT":LSTAT,"MEDV":MEDV})categorical_cols = ["CHAS", "RAD"]continuous_cols = [c for c in df_h.columns if c not in categorical_cols and c != "MEDV"]print("shape:", df_h.shape)df_h.head()

### 7-1. 타깃 기초 통계와 이상치 경계

In [ ]:
print(f"샘플 수 : {df_h.shape[0]}")print(f"특성 수 (타깃 제외) : {df_h.shape[1] - 1}")print(f"\nMEDV 통계")print(f"  평균     : ${df_h['MEDV'].mean():.2f}k")print(f"  중앙값   : ${df_h['MEDV'].median():.2f}k")print(f"  표준편차 : ${df_h['MEDV'].std():.2f}k")print(f"  범위     : ${df_h['MEDV'].min():.1f}k ~ ${df_h['MEDV'].max():.1f}k")def calculate_outlier_bounds(series, k=1.5):    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)    IQR = Q3 - Q1    return Q1 - k*IQR, Q3 + k*IQRlo, hi = calculate_outlier_bounds(df_h["MEDV"])ratio = ((df_h["MEDV"] < lo) | (df_h["MEDV"] > hi)).mean() * 100print(f"\nMEDV 이상치 경계 : [{lo:.2f}, {hi:.2f}]  → 이상치 비율 {ratio:.2f}%")print("결측치 총 개수 :", int(df_h.isnull().sum().sum()))

### 7-2. 상관관계 — 강한 상관 쌍 찾기 (다중공선성 후보)

In [ ]:
dfc = df_h.corr()corr_with_medv = dfc["MEDV"].drop("MEDV")top_corr_feature = corr_with_medv.abs().idxmax()top_corr_value = corr_with_medv[top_corr_feature]print(f"MEDV와 |r| 최대 특성: {top_corr_feature}  (r = {top_corr_value:.4f})")print(corr_with_medv.abs().sort_values(ascending=False).head(5).round(4))strong_corr_pairs = []cols = dfc.columns.tolist()for i in range(len(cols)):    for j in range(i+1, len(cols)):        if abs(dfc.iloc[i, j]) >= 0.7:            strong_corr_pairs.append((cols[i], cols[j], round(dfc.iloc[i, j], 3)))print("\n|r| >= 0.7 인 특성 쌍:", strong_corr_pairs)fig, ax = plt.subplots(figsize=(11, 8.5))mask = np.triu(np.ones_like(dfc), k=1)sns.heatmap(dfc, annot=True, fmt=".2f", cmap="coolwarm", mask=mask,            vmin=-1, vmax=1, center=0, square=True, linewidths=.5,            cbar_kws={"shrink": .8}, annot_kws={"size": 7}, ax=ax)ax.set_title("Correlation Matrix (Housing)")plt.tight_layout(); plt.show()

### 7-3. 전처리 — 분할 → 이상치 제거 → 표준화- 연속형 타깃은 그대로 `stratify` 불가 → `pd.qcut`으로 구간을 나눠 stratify- IQR 경계는 **train에서 계산해 test에도 그대로 적용** (누수 방지)

In [ ]:
from sklearn.model_selection import train_test_splitX = df_h.drop("MEDV", axis=1)y = df_h["MEDV"]y_bins = pd.qcut(y, 10, labels=False, duplicates="drop")   # 연속형 → 10구간X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.3, random_state=42, stratify=y_bins)print(f"Train {len(X_train)}개 / Test {len(X_test)}개")# IQR 경계는 train에서만 계산iqr_bounds = {}for col in continuous_cols:    Q1, Q3 = X_train[col].quantile(.25), X_train[col].quantile(.75)    IQR = Q3 - Q1    iqr_bounds[col] = (Q1 - 1.5*IQR, Q3 + 1.5*IQR)mask_train = pd.Series(True, index=X_train.index)for col, (lo_, hi_) in iqr_bounds.items():    mask_train &= X_train[col].between(lo_, hi_)X_train, y_train = X_train[mask_train], y_train[mask_train]mask_test = pd.Series(True, index=X_test.index)for col, (lo_, hi_) in iqr_bounds.items():          # ← train 경계를 그대로 사용!    mask_test &= X_test[col].between(lo_, hi_)X_test, y_test = X_test[mask_test], y_test[mask_test]print(f"이상치 제거 후 Train {len(X_train)}개 / Test {len(X_test)}개")scaler = StandardScaler()X_train = X_train.copy(); X_test = X_test.copy()X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])X_test[continuous_cols]  = scaler.transform(X_test[continuous_cols])print(f"연속형 평균(≈0)   : {X_train[continuous_cols].mean().mean():.6f}")print(f"연속형 표준편차(≈1): {X_train[continuous_cols].std().mean():.6f}")

### 7-4. 선형 회귀 학습 및 평가$$\hat y=\theta_0+\theta_1x_1+\cdots+\theta_nx_n$$- sklearn의 `LinearRegression`은 정규방정식/SVD로 **한 번에** 최적 해를 구한다 (반복 학습 없음)- RMSE: 큰 오차에 민감 / MAE: 이상치에 둔감 / R²: 설명력

In [ ]:
from sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_absolute_error, r2_scoretry:    from sklearn.metrics import root_mean_squared_error    def rmse_fn(a, b): return root_mean_squared_error(a, b)except ImportError:                                   # sklearn < 1.4    from sklearn.metrics import mean_squared_error    def rmse_fn(a, b): return mean_squared_error(a, b) ** 0.5model = LinearRegression()model.fit(X_train, y_train)y_pred = model.predict(X_test)rmse = rmse_fn(y_test, y_pred)mae  = mean_absolute_error(y_test, y_pred)r2   = r2_score(y_test, y_pred)print("=== 모델 성능 (Test Set) ===")print(f"RMSE : ${rmse:.3f}k")print(f"MAE  : ${mae:.3f}k")print(f"R²   : {r2:.4f}")print(f"\nRMSE/MAE = {rmse/mae:.3f}  (1에 가까우면 오차가 고르게 퍼진 것)")coef = pd.Series(model.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)print("\n영향력 큰 계수 top5 (표준화된 특성 기준)")print(coef.head(5).round(4))

In [ ]:
# 훈련 성능과 비교 → 과적합 여부 확인print(f"Train R² : {model.score(X_train, y_train):.4f}")print(f"Test  R² : {model.score(X_test, y_test):.4f}")print("두 값이 비슷하면 과적합이 심하지 않다는 신호")

### 7-5. 예측 결과 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4), ncols=2)# (1) 실제 vs 예측ax[0].scatter(y_test, y_pred, alpha=.6)lo_, hi_ = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())ax[0].plot([lo_, hi_], [lo_, hi_], "r--", lw=2, label="Perfect Prediction")ax[0].set_xlabel("Actual MEDV ($1000s)"); ax[0].set_ylabel("Predicted MEDV ($1000s)")ax[0].set_title(f"Actual vs Predicted\nRMSE={rmse:.3f}, R2={r2:.4f}")ax[0].legend(); ax[0].grid(alpha=.3)# (2) 잔차 플롯 — 패턴이 없어야 좋다resid = y_test - y_predax[1].scatter(y_pred, resid, alpha=.6, color="coral")ax[1].axhline(0, color="red", ls="--")ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Residual (y - ŷ)")ax[1].set_title("Residual Plot"); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()print("해석:")print("- 점들이 빨간 선(y=x)에 가까울수록 예측이 정확")print("- 선 위쪽: 과대 예측 / 아래쪽: 과소 예측")print("- 잔차 플롯에 곡선·깔때기 모양이 보이면 선형 가정이나 등분산 가정이 깨진 신호")

---## Step 8. Pipeline — 누수를 구조적으로 막기전처리와 모델을 하나로 묶으면 `cross_val_score` 안에서도 **각 fold의 train으로만** 스케일러가 fit 됩니다.

In [ ]:
from sklearn.pipeline import Pipelinepipe = Pipeline([    ("imputer", SimpleImputer(strategy="median")),    ("scaler",  StandardScaler()),    ("clf",     LogisticRegression(max_iter=1000)),])# Wine 이진 분류로 다시 (원본 X, y 사용 — 전처리는 파이프라인이 알아서)Xw = df.drop("quality", axis=1).valuesyw = (df["quality"].values != 0).astype(int)Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=.3, random_state=42, stratify=yw)pipe.fit(Xw_tr, yw_tr)print("Test accuracy:", round(pipe.score(Xw_te, yw_te), 4))scores = cross_val_score(pipe, Xw_tr, yw_tr, cv=5, scoring="f1")print("CV F1 (누수 없음):", np.round(scores, 4), "→ 평균", round(scores.mean(), 4))

---## 마무리 — 전체 흐름 요약```데이터 로딩   ↓EDA        : shape · describe · corr/heatmap · histplot · scatter/pairplot   ↓품질 점검  : isnull().sum() · IQR 이상치 · 왜 생겼는지 확인   ↓분할       : train_test_split(test_size, random_state, stratify)   ← 반드시 먼저!   ↓전처리     : SimpleImputer · StandardScaler  (train으로 fit, test는 transform)   ↓학습       : LogisticRegression / LinearRegression  .fit()   ↓평가       : 분류 → confusion_matrix · classification_report · ROC-AUC             회귀 → RMSE · MAE · R² · 잔차 플롯   ↓검증·해석  : cross_val_score · PCA · KMeans```**다음:** `22_문제집.ipynb` 로 직접 구현해 보세요. 정답은 `23_해답.ipynb`.